# Audio → LLM POC — Colab Trainer

This notebook pulls `Kesehet/audio-llm-poc`, validates the Colab runtime, prepares public speech data, and starts Stage-1 projector training.

**Recommended runtime:** GPU. A T4 is enough for the starter run; L4/A100 is better.

Stage 1 objective:

`audio → frozen Whisper encoder → trainable projector → frozen Qwen → transcription`

The notebook includes recovery checks so **Runtime → Run all** is the intended path.


In [ ]:
import os, shutil, subprocess, sys, platform
from pathlib import Path

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())

# Recover from the common Colab state where the previous working directory
# was deleted by a repo refresh.
try:
    print("Initial working directory:", os.getcwd())
except FileNotFoundError:
    os.chdir("/content")
    print("Recovered invalid working directory ->", os.getcwd())

os.chdir("/content")
print("Safe working directory:", os.getcwd())


In [ ]:
import torch

subprocess.run(["nvidia-smi"], check=False)

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU detected. In Colab choose Runtime → Change runtime type → GPU, then Run all again."
    )

gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
print("GPU:", gpu_name)
print("VRAM GB:", round(vram_gb, 1))


## Refresh the repository safely

Important: we first move to `/content`, **then** remove the old clone. This avoids deleting Colab's current working directory underneath Python.


In [ ]:
import os, shutil, subprocess

REPO_DIR = Path("/content/audio-llm-poc")
REPO_URL = "https://github.com/Kesehet/audio-llm-poc.git"

os.chdir("/content")

if REPO_DIR.exists():
    print("Removing previous clone:", REPO_DIR)
    shutil.rmtree(REPO_DIR)

subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)

print("Current directory:", os.getcwd())
subprocess.run(["git", "log", "-1", "--oneline"], check=True)

required = [
    "scripts/prepare_public_asr.py",
    "scripts/merge_manifests.py",
    "train.py",
    "configs/poc.yaml",
    "requirements.txt",
]
missing = [p for p in required if not Path(p).exists()]
if missing:
    raise RuntimeError(f"Repo clone is incomplete. Missing: {missing}")

print("Repository check: OK")


In [ ]:
# Install project dependencies into the current Colab runtime.
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
    check=True,
)
print("Dependencies installed.")


## Dataset smoke test

Before downloading hundreds of files, the notebook prepares **10 English FLEURS samples**. If anything is wrong with Hugging Face access or audio decoding, this cell prints the underlying stdout/stderr instead of hiding it behind `CalledProcessError`.


In [ ]:
import os, shutil, subprocess, sys
from pathlib import Path

os.chdir("/content/audio-llm-poc")

smoke_dir = Path("data/smoke")
if smoke_dir.exists():
    shutil.rmtree(smoke_dir)

cmd = [
    sys.executable,
    "scripts/prepare_public_asr.py",
    "fleurs-en",
    "--limit", "10",
    "--output", str(smoke_dir),
]

print("Running smoke test:")
print(" ".join(cmd))

result = subprocess.run(cmd, text=True, capture_output=True)

print("\n--- stdout ---")
print(result.stdout)
print("--- stderr ---")
print(result.stderr)

if result.returncode != 0:
    raise RuntimeError(
        f"Dataset smoke test failed with exit code {result.returncode}. "
        "The full stdout/stderr is printed above."
    )

manifest = smoke_dir / "fleurs-en" / "manifest.jsonl"
rows = manifest.read_text(encoding="utf-8").strip().splitlines() if manifest.exists() else []
if len(rows) != 10:
    raise RuntimeError(f"Expected 10 smoke-test rows, found {len(rows)}")

print("Dataset smoke test: OK — 10/10 examples prepared.")


## Choose starter dataset size

The default is intentionally small so the first run proves the full pipeline before spending hours on training.

Set `FAST_TEST = False` later for the larger starter corpus.


In [ ]:
FAST_TEST = True

if FAST_TEST:
    limits = {
        "fleurs-en": 150,
        "fleurs-hi": 150,
        "librispeech-clean": 300,
        "voxpopuli-en": 300,
    }
else:
    limits = {
        "fleurs-en": 1000,
        "fleurs-hi": 1000,
        "librispeech-clean": 3000,
        "voxpopuli-en": 3000,
    }

print("Dataset limits:", limits)
print("Expected maximum rows:", sum(limits.values()))


In [ ]:
import os, shutil, subprocess, sys
from pathlib import Path

os.chdir("/content/audio-llm-poc")

public_dir = Path("data/public")
if public_dir.exists():
    shutil.rmtree(public_dir)

for dataset_name, limit in limits.items():
    print(f"\n{'='*70}\n{dataset_name}: {limit} samples\n{'='*70}")
    cmd = [
        sys.executable,
        "scripts/prepare_public_asr.py",
        dataset_name,
        "--limit", str(limit),
    ]
    result = subprocess.run(cmd, text=True)
    if result.returncode != 0:
        raise RuntimeError(
            f"{dataset_name} failed with exit code {result.returncode}. "
            "Scroll immediately above this message for the dataset error."
        )

print("\nAll requested datasets prepared.")


In [ ]:
import os, subprocess, sys
from pathlib import Path

os.chdir("/content/audio-llm-poc")

manifests = [
    "data/public/fleurs-en/manifest.jsonl",
    "data/public/fleurs-hi/manifest.jsonl",
    "data/public/librispeech-clean/manifest.jsonl",
    "data/public/voxpopuli-en/manifest.jsonl",
]

for manifest in manifests:
    p = Path(manifest)
    if not p.exists():
        raise FileNotFoundError(f"Missing manifest: {manifest}")
    count = sum(1 for line in p.open(encoding="utf-8") if line.strip())
    print(f"{manifest}: {count} rows")

subprocess.run(
    [
        sys.executable,
        "scripts/merge_manifests.py",
        *manifests,
        "--output", "data/stage1.jsonl",
    ],
    check=True,
)

stage1 = Path("data/stage1.jsonl")
stage1_rows = sum(1 for line in stage1.open(encoding="utf-8") if line.strip())
print("\nMerged Stage-1 rows:", stage1_rows)

if stage1_rows == 0:
    raise RuntimeError("Merged training manifest is empty.")

print("\nFirst two manifest rows:")
with stage1.open(encoding="utf-8") as f:
    for i, line in enumerate(f):
        print(line.rstrip())
        if i >= 1:
            break


## Colab training config

This uses FP16 on T4-class GPUs, batch size 1, and trains only the projector. For very small GPUs it automatically falls back from Whisper-small to Whisper-base.


In [ ]:
from pathlib import Path
import os, yaml, torch

os.chdir("/content/audio-llm-poc")

cfg = yaml.safe_load(Path("configs/poc.yaml").read_text(encoding="utf-8"))
cfg["batch_size"] = 1
cfg["grad_accum_steps"] = 8
cfg["epochs"] = 3 if FAST_TEST else 2
cfg["output_dir"] = "checkpoints/colab-stage1"

if torch.cuda.is_bf16_supported():
    cfg["mixed_precision"] = "bf16"
else:
    cfg["mixed_precision"] = "fp16"

vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
if vram_gb < 13:
    cfg["audio_encoder"] = "openai/whisper-base"
    print("Low VRAM detected: using Whisper-base.")

Path("configs/colab-runtime.yaml").write_text(
    yaml.safe_dump(cfg, sort_keys=False),
    encoding="utf-8",
)

print(Path("configs/colab-runtime.yaml").read_text(encoding="utf-8"))


## Start training

This is the first expensive step. Everything before it is validation/data preparation.


In [ ]:
import os, subprocess, sys, time

os.chdir("/content/audio-llm-poc")

print("Starting Stage-1 training...")
start = time.time()

result = subprocess.run(
    [
        sys.executable,
        "train.py",
        "--manifest", "data/stage1.jsonl",
        "--config", "configs/colab-runtime.yaml",
    ]
)

elapsed = (time.time() - start) / 60
print(f"Training process finished after {elapsed:.1f} minutes with exit code {result.returncode}.")

if result.returncode != 0:
    raise RuntimeError("Training failed. Scroll immediately above for the underlying error.")


## Check saved projector weights


In [ ]:
from pathlib import Path
import os

os.chdir("/content/audio-llm-poc")

checkpoints = sorted(Path("checkpoints/colab-stage1").glob("projector-epoch-*.pt"))
if not checkpoints:
    raise RuntimeError("Training completed without creating projector checkpoints.")

print("Saved checkpoints:")
for p in checkpoints:
    print(" -", p, f"({p.stat().st_size / 1024**2:.1f} MB)")


## Optional: save checkpoints to Google Drive

Run this after training if you want the weights to survive the Colab runtime.


In [ ]:
# Uncomment to save to Drive.
# from google.colab import drive
# drive.mount('/content/drive')
# !mkdir -p /content/drive/MyDrive/audio-llm-poc-checkpoints
# !cp -v checkpoints/colab-stage1/*.pt /content/drive/MyDrive/audio-llm-poc-checkpoints/
